In [1]:
import json
import os
import shutil
import random
from pathlib import Path


old_label_map = {
    '자전거 도로(바닥표시)': 0,  # bike lane - removed, not in final 4
    '12.맨홀': 1,
    '맨홀':    1,
    '23.방치물(전동킥보드)': 2,
    '방치물(전동킥보드)':    2,
}




old_label_map = {
    '23.방치물(전동킥보드)': 0,  # kickboard
    '방치물(전동킥보드)':    0,  # kickboard
    '12.맨홀':              1,  # manhole
    '맨홀':                 1,  # manhole
}


dataset_pairs = [
    (r"C:\Users\konyang\Dataset\kickboard_image",  r"C:\Users\konyang\Dataset\kickboard_label",  'old'),
    (r"C:\Users\konyang\Dataset\manhole_image",    r"C:\Users\konyang\Dataset\manhole_label",    'old'),
    (r"C:\Users\konyang\Dataset\기둥 이미지",        r"C:\Users\konyang\Dataset\기둥 라벨링",        'new'),
    (r"C:\Users\konyang\Dataset\소화전 이미지",      r"C:\Users\konyang\Dataset\소화전 라벨링",      'new'),
]


output_dir = r"C:\Users\konyang\Dataset\detect_dataset"
for split in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)


random.seed(42)
converted = 0
skipped = 0

for img_dir, lbl_dir, fmt in dataset_pairs:
    json_files = [f for f in os.listdir(lbl_dir) if f.endswith('.json')]

    for fname in json_files:
        with open(os.path.join(lbl_dir, fname), 'r', encoding='utf-8') as f:
            data = json.load(f)

        lines = []

        
        if fmt == 'old':
            img_info   = data['images']
            img_width  = img_info['width']
            img_height = img_info['height']
            img_name   = img_info['filename']

            for ann in data['annotation']:
                label = ann['label']
                if label not in old_label_map:
                    continue
                class_id = old_label_map[label]

                if 'segmentation' in ann:
                    pts = ann['segmentation']
                    xs = [p['x'] for p in pts]
                    ys = [p['y'] for p in pts]
                    x_min, x_max = min(xs), max(xs)
                    y_min, y_max = min(ys), max(ys)
                elif 'bbox' in ann:
                    x_min, y_min, bw, bh = ann['bbox']
                    x_max = x_min + bw
                    y_max = y_min + bh
                else:
                    continue

                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width    = (x_max - x_min) / img_width
                height   = (y_max - y_min) / img_height

                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                width    = max(0, min(1, width))
                height   = max(0, min(1, height))

                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        
        elif fmt == 'new':
            img_info   = data['info']
            img_width  = img_info['width']
            img_height = img_info['height']
            img_name   = img_info['filename']

            
            if '기둥' in lbl_dir:
                class_id = 2  # bollard
            else:
                class_id = 3  # fire hydrant

            for ann in data['annotations']:
                bbox = ann['annotation_info'][0]  # [x_min, y_min, x_max, y_max]
                x_min, y_min, x_max, y_max = bbox

                x_center = ((x_min + x_max) / 2) / img_width
                y_center = ((y_min + y_max) / 2) / img_height
                width    = (x_max - x_min) / img_width
                height   = (y_max - y_min) / img_height

                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                width    = max(0, min(1, width))
                height   = max(0, min(1, height))

                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        if not lines:
            skipped += 1
            continue

        # 80/20 train/val split
        split = 'train' if random.random() < 0.8 else 'val'

        
        src_img = os.path.join(img_dir, img_name)
        dst_img = os.path.join(output_dir, f'images/{split}', img_name)
        if not os.path.exists(src_img):
            skipped += 1
            continue
        shutil.copy(src_img, dst_img)

       
        txt_name = Path(img_name).stem + '.txt'
        dst_lbl  = os.path.join(output_dir, f'labels/{split}', txt_name)
        with open(dst_lbl, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))

        converted += 1

print(f"Converted: {converted} | Skipped: {skipped}")

✅ Done! Converted: 2627 | Skipped: 3373


In [2]:
yaml_content = """path: C:/Users/konyang/Dataset/detect_dataset
train: images/train
val: images/val

nc: 4
names: ['kickboard', 'manhole', 'bollard', 'fire_hydrant']
"""

with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created!")

✅ data.yaml created!


In [3]:
import os

output_dir = r"C:\Users\konyang\Dataset\detect_dataset"

train_imgs = len(os.listdir(os.path.join(output_dir, 'images/train')))
val_imgs   = len(os.listdir(os.path.join(output_dir, 'images/val')))
train_lbls = len(os.listdir(os.path.join(output_dir, 'labels/train')))
val_lbls   = len(os.listdir(os.path.join(output_dir, 'labels/val')))

print(f"Train images: {train_imgs} | Train labels: {train_lbls}")
print(f"Val images:   {val_imgs}   | Val labels:   {val_lbls}")

Train images: 2133 | Train labels: 2133
Val images:   494   | Val labels:   494


In [5]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

model.train(
    data=r'C:\Users\konyang\Desktop\detect_dataset\data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
)

New https://pypi.org/project/ultralytics/8.4.58 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.13.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\konyang\Desktop\detect_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.9

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F9CB33DD30>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [9]:
from ultralytics import YOLO

model = YOLO('yolo26s.pt')

model.train(
    data=r'C:\Users\konyang\Desktop\detect_dataset\data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=0,

    project=r'C:\Users\konyang\Desktop',
    name='yolo26sdete100ep',
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.13.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\konyang\Desktop\detect_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F912745B70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [1]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

model.train(
    data=r'C:\Users\konyang\Desktop\detect_dataset\data.yaml',
    epochs=70,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,

    project=r'C:\Users\konyang\runs\detect',
    name='yolo26n_70ep',
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.13.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\konyang\Desktop\detect_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.93

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001820CB9A820>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [2]:
from ultralytics import YOLO

model = YOLO('yolo26s.pt')

model.train(
    data=r'C:\Users\konyang\Desktop\detect_dataset\data.yaml',
    epochs=70,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,

    project=r'C:\Users\konyang\runs\detect',
    name='yolo26s_70ep',
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.13.9 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\konyang\Desktop\detect_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.93

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001820ABBC1A0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    